In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


In [12]:
df = pd.read_csv('../data/target/access_log_master.csv')

print(df.shape)
df.head()


(1054606, 31)


,ip_client,timestamp,status,size,user_agent,method,url,protocol,anomaly,status_category,...,url__count_percentage_symbol,url__count_question_symbol,url__count_hyphen,url__count_equal,url__url_length,url__digit_count,url__letter_count,url__count_special_characters,url__is_encoded,url__unusual_character_ratio
0,47.128.121.63,2025-11-25 00:00:16-05:00,206,500,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/2021/06/b2ap3_lar...,HTTP/2.0,-1,200,...,0,0,2,0,74,10,52,12,0,0.094595
1,47.128.121.63,2025-11-25 00:00:16-05:00,200,675,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/plugins/elementor/assets/lib/font-...,HTTP/2.0,-1,200,...,0,1,2,1,83,4,63,16,0,0.120482
2,47.128.121.63,2025-11-25 00:00:16-05:00,200,264,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/elementor/css/pos...,HTTP/2.0,-1,200,...,0,1,2,1,70,15,43,12,0,0.128571
3,47.128.121.63,2025-11-25 00:00:16-05:00,206,500,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/2021/05/73192906_...,HTTP/2.0,-1,200,...,0,0,2,0,96,57,26,13,0,0.072917
4,47.128.121.63,2025-11-25 00:00:16-05:00,206,500,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/2021/07/PNG-image...,HTTP/2.0,-1,200,...,0,0,4,0,61,16,33,12,0,0.114754


In [11]:
# columns_to_drop = [
#     "ip_client",
#     "timestamp",
#     "user_agent",
#     "url",
#     "protocol",
#     "anomaly"  # importante: no entrenar con etiquetas
# ]

# df_model = df.drop(columns=columns_to_drop, errors="ignore")
# df_model.head()
# 1. Eliminar columnas de alta cardinalidad peligrosa
columns_to_drop = [
    "ip_client",
    "timestamp",
    "user_agent",
    "url",
    "anomaly"
]

df_clean = df.drop(columns=columns_to_drop, errors="ignore")

# 2. One-hot solo donde tiene sentido
categorical_cols = ["method", "protocol", "status_category"]

df_encoded = pd.get_dummies(
    df_clean,
    columns=categorical_cols,
    drop_first=True
)
print(df_encoded.shape)
df_encoded.head()



(1054606, 71)


,status,size,url__count_sql_words,url__count_xss_words,url__count_command_words,url__count_auth_words,url__count_error_words,url__count_malware_words,url__count_danger_characters,url__count_obfuscation_code_words,...,"protocol_\xB7#\x83x`)\x09\xA2\x02X\xAE\x1A\xA8F\x84\xE4_.\x93\x14\xFF\xD1\x08\xB4\xAE\xB2\x82\x02\x7F\x00L\xFB\x00&\xCC\xA8\xCC\xA9\xC0/\xC00\xC0+\xC0,\xC0\x13\xC0\x09\xC0\x14\xC0","protocol_\xD1Y7n\xFFG\xDEo\x5C\x8D8\xA8\xCB\xA6f\x0Ba\xC9\x07\xF9\x0B\xFF)y\xA8>\x5Cop+\xADW\x00\x1A\xC0+\xC0/\xC0,\xC00\xCC\xA9\xCC\xA8\xC0\x09\xC0\x13\xC0","protocol_\xE1\xEE\xCE\xCERTQ\x1E\xFE\xCE\xC2\xD5\x9D\x02^mS(\xC8WU\x88\xB9%\x85s`\xDA\x00\x9C\x13\x02\x13\x03\x13\x01\x003\x009\x005\x00/\xC0,\xC00\x00\xA3\x00\x9F\xCC\xA9\xCC\xA8\xCC\xAA\xC0\xAF\xC0\xAD\xC0\xA3\xC0\x9F\xC0]\xC0a\xC0W\xC0S\xC0+\xC0/\x00\xA2\x00\x9E\xC0\xAE\xC0\xAC\xC0\xA2\xC0\x9E\xC0\x5C\xC0`\xC0V\xC0R\xC0$\xC0(\x00k\x00j\xC0s\xC0w\x00\xC4\x00\xC3\xC0#\xC0'\x00g\x00@\xC0r\xC0v\x00\xBE\x00\xBD\xC0","protocol_\xED\x85\x01l?c<\xD7.\xD4\xE6-b\xE4I\x00&\xCC\xA8\xCC\xA9\xC0/\xC00\xC0+\xC0,\xC0\x13\xC0\x09\xC0\x14\xC0",protocol_a\xE3\x93,protocol_encoding=\x22UTF-8\x22?>,status_category_200,status_category_300,status_category_400,status_category_500
0,206,500,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,False
1,200,675,1,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,False
2,200,264,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,False
3,206,500,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,False
4,206,500,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,False


In [ ]:
features = [
    "status",
    "size",
    "method",
    "status_category",
    "url__count_sql_words",
    "url__count_xss_words",
    "url__count_command_words",
    "url__count_auth_words",
    "url__count_error_words",
    "url__count_malware_words",
    "url__count_danger_characters",
    "url__count_obfuscation_code_words",
    "url__count_dir_words",
    "url__count_dot",
    "url__count_http",
    "url__count_percentage_symbol",
    "url__count_question_symbol",
    "url__count_hyphen",
    "url__count_equal",
    "url__url_length",
    "url__digit_count",
    "url__letter_count",
    "url__count_special_characters",
    "url__is_encoded",
    "url__unusual_character_ratio"
]

X = df_encoded[features]
X.head()


KeyError: "['method', 'status_category'] not in index"

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


ValueError: could not convert string to float: 'GET'

In [ ]:
iso_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_scaled)


In [ ]:
df["iforest_score"] = iso_forest.decision_function(X_scaled)
df["iforest_pred"] = iso_forest.predict(X_scaled)

# Convención sklearn:
#  1  -> normal
# -1  -> anomalía

df["is_anomaly_iforest"] = (df["iforest_pred"] == -1).astype(int)

df[["iforest_score", "is_anomaly_iforest"]].head()


In [ ]:
total = len(df)
anomalies = df["is_anomaly_iforest"].sum()

print(f"Total requests: {total}")
print(f"Anomalías detectadas: {anomalies}")
print(f"Porcentaje: {anomalies / total:.2%}")


In [ ]:
df_anomalies = (
    df[df["is_anomaly_iforest"] == 1]
    .sort_values("iforest_score")
)

df_anomalies[
    [
        "ip_client",
        "method",
        "status",
        "url__url_length",
        "url__count_sql_words",
        "url__count_xss_words",
        "url__count_danger_characters",
        "url__is_encoded",
        "url__unusual_character_ratio",
        "iforest_score"
    ]
].head(20)


In [ ]:
df.groupby("is_anomaly_iforest").mean()[
    [
        "url__url_length",
        "url__count_sql_words",
        "url__count_xss_words",
        "url__count_danger_characters",
        "url__unusual_character_ratio"
    ]
]
